# Postgres Data Explorer

Use this notebook to inspect the normalized source data and processed outputs in Azure Postgres. It is read-only: every helper below runs `select` queries only.

Set the filters in the setup cell, then run the overview and detail sections as needed.

## Setup

Connection order:

1. Use `AZURE_POSTGRES_DSN` from the notebook environment.
2. If that is missing, use Azure CLI to read `AZURE_POSTGRES_DSN` from the Processing Function App settings.

If imports fail, run the install cell once.


If the connection times out after the DSN loads, Azure Postgres is reachable from Azure but your local IP is probably not in the server firewall rules. Add a temporary firewall rule for your current public IP or run the notebook from an Azure-hosted environment.

In [ ]:
# Run once if your kernel is missing packages.
# %pip install pandas psycopg[binary]

In [1]:
from __future__ import annotations

import os
import re
import subprocess
from typing import Any

import pandas as pd
import psycopg
from psycopg.rows import dict_row

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "eliza-ai-workbench-rg")
PROCESSING_FUNCTION_APP = os.getenv(
    "AZURE_PROCESSING_FUNCTION_APP",
    "eliza-ai-workbench-processing-func-18878",
)

TICKER = "AMD"          # Try: AMD, SNDK, FROG, APP, KVYO, or None
SOURCE = None           # Try: reddit, hacker_news, github, or None
LOOKBACK_DAYS = 90      # summaries and connections currently use 90 days
LIMIT = 25


def _az_command() -> str:
    candidates = [
        "az",
        r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd",
    ]
    for candidate in candidates:
        try:
            subprocess.run([candidate, "--version"], check=True, capture_output=True, text=True)
            return candidate
        except Exception:
            continue
    raise RuntimeError("Azure CLI was not found. Set AZURE_POSTGRES_DSN in the notebook environment instead.")


def load_postgres_dsn() -> str:
    dsn = os.getenv("AZURE_POSTGRES_DSN")
    if dsn:
        return dsn

    az = _az_command()
    command = [
        az,
        "functionapp",
        "config",
        "appsettings",
        "list",
        "--name",
        PROCESSING_FUNCTION_APP,
        "--resource-group",
        RESOURCE_GROUP,
        "--query",
        "[?name=='AZURE_POSTGRES_DSN'].value | [0]",
        "-o",
        "tsv",
    ]
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    dsn = result.stdout.strip()
    if not dsn:
        raise RuntimeError("AZURE_POSTGRES_DSN was not found in the Function App settings.")
    if dsn.startswith("@Microsoft.KeyVault("):
        match = re.search(r"SecretUri=([^)]*)", dsn)
        if not match:
            raise RuntimeError("AZURE_POSTGRES_DSN is a Key Vault reference, but no SecretUri was found.")
        secret_uri = match.group(1)
        secret = subprocess.run(
            [az, "keyvault", "secret", "show", "--id", secret_uri, "--query", "value", "-o", "tsv"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
        if not secret:
            raise RuntimeError("Key Vault returned an empty AZURE_POSTGRES_DSN secret.")
        return secret
    return dsn


DSN = load_postgres_dsn()
print("Loaded Postgres connection. DSN hidden.")

Loaded Postgres connection. DSN hidden.


## Query Helpers

In [2]:
def read_sql(sql: str, params: tuple[Any, ...] | None = None) -> pd.DataFrame:
    """Run a read-only query and return a DataFrame."""
    normalized = sql.strip().lower()
    if not normalized.startswith(("select", "with")):
        raise ValueError("This notebook only allows read-only SELECT queries.")
    with psycopg.connect(DSN, row_factory=dict_row) as conn:
        rows = conn.execute(sql, params or ()).fetchall()
    return pd.DataFrame([dict(row) for row in rows])


def _filters(alias: str = "") -> tuple[str, list[Any]]:
    prefix = f"{alias}." if alias else ""
    clauses: list[str] = []
    params: list[Any] = []
    if TICKER:
        clauses.append(f"{prefix}ticker = %s")
        params.append(TICKER)
    if SOURCE:
        clauses.append(f"{prefix}source = %s")
        params.append(SOURCE)
    return ("where " + " and ".join(clauses), params) if clauses else ("", params)


def show(df: pd.DataFrame, rows: int = 25) -> pd.DataFrame:
    return df.head(rows)

## Database Overview

In [3]:
tables = [
    "source_items",
    "dataload_runs",
    "processing_runs",
    "item_enrichments",
    "item_embeddings",
    "item_connections",
    "brain_summaries",
    "sentiment_weekly",
]

count_sql = " union all ".join(f"select '{table}' as table_name, count(*)::int as rows from {table}" for table in tables)
read_sql(count_sql).sort_values("table_name")

ConnectionTimeout: connection timeout expired

In [ ]:
read_sql(
    """
    select table_name, column_name, data_type, udt_name, is_nullable
    from information_schema.columns
    where table_schema = 'public'
      and table_name = any(%s)
    order by table_name, ordinal_position
    """,
    (tables,),
)

## Source Data

Raw payloads live in Blob Storage. `source_items` is the normalized Postgres table used by processing.

In [ ]:
read_sql(
    """
    select ticker, source,
           count(*)::int as source_items,
           min(published_at) as earliest_published_at,
           max(published_at) as latest_published_at,
           max(fetched_at) as latest_fetched_at
    from source_items
    group by ticker, source
    order by ticker, source
    """
)

In [ ]:
where, params = _filters()
params.append(LIMIT)
read_sql(
    f"""
    select source_item_id, ticker, source, published_at, fetched_at,
           title, left(body, 800) as body_preview, source_url, author, metadata
    from source_items
    {where}
    order by published_at desc nulls last, fetched_at desc
    limit %s
    """,
    tuple(params),
)

## Processing Coverage

This summarizes source items, enrichments, embeddings, connections, summaries, and sentiment rows by ticker.

In [ ]:
read_sql(
    """
    with source_counts as (
        select ticker, count(*)::int as source_items from source_items group by ticker
    ), enrich_counts as (
        select ticker, count(*)::int as enriched_items,
               count(*) filter (where relevance >= 0)::int as relevant_items,
               max(relevance)::int as max_relevance
        from item_enrichments group by ticker
    ), embedding_counts as (
        select ticker, count(*)::int as embedded_items from item_embeddings group by ticker
    ), connection_counts as (
        select ticker, count(*)::int as connections_total,
               count(*) filter (where valid = true and confidence >= 0.25)::int as valid_connections
        from item_connections group by ticker
    ), summary_counts as (
        select ticker, count(*)::int as summaries from brain_summaries group by ticker
    ), sentiment_counts as (
        select ticker, count(*)::int as sentiment_rows from sentiment_weekly group by ticker
    )
    select sc.ticker,
           sc.source_items,
           coalesce(ec.enriched_items, 0) as enriched_items,
           coalesce(ec.relevant_items, 0) as relevant_items,
           coalesce(emc.embedded_items, 0) as embedded_items,
           coalesce(cc.connections_total, 0) as connections_total,
           coalesce(cc.valid_connections, 0) as valid_connections,
           coalesce(suc.summaries, 0) as summaries,
           coalesce(sec.sentiment_rows, 0) as sentiment_rows,
           ec.max_relevance
    from source_counts sc
    left join enrich_counts ec using (ticker)
    left join embedding_counts emc using (ticker)
    left join connection_counts cc using (ticker)
    left join summary_counts suc using (ticker)
    left join sentiment_counts sec using (ticker)
    order by sc.ticker
    """
)

## Enriched Items

In [ ]:
where, params = _filters("ie")
params.append(LIMIT)
read_sql(
    f"""
    select ie.source_item_id, ie.ticker, ie.source, si.published_at,
           ie.relevance, ie.sentiment, ie.sentiment_rationale,
           ie.themes, ie.firsthand, ie.firsthand_type,
           ie.summary, si.title, si.source_url, si.metadata, ie.model, ie.enriched_at
    from item_enrichments ie
    join source_items si on si.source_item_id = ie.source_item_id
    {where}
    order by ie.relevance desc, si.published_at desc nulls last
    limit %s
    """,
    tuple(params),
)

## Embeddings

In [ ]:
where, params = _filters("emb")
params.append(LIMIT)
read_sql(
    f"""
    select emb.source_item_id, emb.ticker, emb.source, emb.published_at,
           vector_dims(emb.embedding) as embedding_dimensions,
           emb.model, emb.embedded_at,
           ie.relevance, ie.sentiment, ie.themes,
           emb.summary
    from item_embeddings emb
    join item_enrichments ie on ie.source_item_id = emb.source_item_id
    {where}
    order by emb.embedded_at desc
    limit %s
    """,
    tuple(params),
)

## Connections And Summaries

These cells apply the same 90-day window used by the deployed synthesis pipeline.

In [ ]:
clauses = ["ic.valid = true", "ic.confidence >= 0.25"]
params: list[Any] = []
if TICKER:
    clauses.append("ic.ticker = %s")
    params.append(TICKER)
where = "where " + " and ".join(clauses)
params.extend([LOOKBACK_DAYS, LOOKBACK_DAYS, LIMIT])

read_sql(
    f"""
    select ic.ticker, ic.confidence, ic.similarity, ic.connection_type,
           ic.source_a, ic.source_b, ic.item_a_id, ic.item_b_id,
           a.published_at as item_a_published_at,
           b.published_at as item_b_published_at,
           ic.narrative, ic.stock_relevance,
           a.title as item_a_title, b.title as item_b_title,
           a.source_url as item_a_url, b.source_url as item_b_url,
           ic.verified_at
    from item_connections ic
    join source_items a on a.source_item_id = ic.item_a_id
    join source_items b on b.source_item_id = ic.item_b_id
    {where}
      and (a.published_at is null or a.published_at >= now() - %s * interval '1 day')
      and (b.published_at is null or b.published_at >= now() - %s * interval '1 day')
    order by ic.confidence desc, ic.similarity desc, ic.verified_at desc
    limit %s`
    """,
    tuple(params),
)

In [7]:
where = "where ticker = %s" if TICKER else ""
params = [TICKER, LIMIT] if TICKER else [LIMIT]
df = read_sql(
    f"""
    select summary_id, ticker, headline, key_signals, cross_source_connections,
           bear_case, confidence, cited_item_ids, invalid_citation_ids,
           model, run_id, generated_at
    from brain_summaries
    {where}
    order by generated_at desc
    limit %s
    """,
    tuple(params),
)

In [12]:
df["key_signals"].iloc[0]

["A developer open-sourced a serving configuration for Kimi K2.6 achieving a 5.6x throughput jump (90 to 508 tok/s) on 8x AMD MI300X GPUs with linear scaling, constant slot latency, and no quality loss, packaged with a reproducible Dockerfile and benchmark utility (hn_comment:db7cf87a81892bf830ee22f6, hn_story:ea6e197104e37ac7cf1558d3). This is firsthand, reproducible evidence that MI300X can serve frontier-class LLMs at production throughput, directly supporting AMD's datacenter AI adoption thesis and lowering the operator barrier to switch from NVIDIA for high-throughput inference. The main qualifier is that BF16 is the only stable precision (FP8 blocked by a model-specific constraint) and validation is currently driven by technical early adopters rather than confirmed enterprise budget commitments.",
 "Across at least eight independent GitHub issues, firsthand developers document critical ROCm reliability regressions on RDNA3/RDNA4 consumer and prosumer cards: idle clock-pinning at 

## Sentiment And Runs

In [ ]:
where, params = _filters()
read_sql(
    f"""
    select ticker, source, week_start, item_count, sentiment_avg,
           rolling_mean_8w, rolling_stddev_8w, z_score, alert, refreshed_at
    from sentiment_weekly
    {where}
    order by ticker, source, week_start desc
    """,
    tuple(params),
)

In [ ]:
read_sql(
    """
    select run_id, status, started_at, completed_at, error_message, metadata
    from processing_runs
    order by started_at desc
    limit %s
    """,
    (LIMIT,),
)

## Trace One Item

Set `SOURCE_ITEM_ID` to inspect a record from source item through enrichment, embedding, and connections.

In [ ]:
SOURCE_ITEM_ID = ""

if SOURCE_ITEM_ID:
    display(read_sql("select * from source_items where source_item_id = %s", (SOURCE_ITEM_ID,)))
    display(read_sql("select * from item_enrichments where source_item_id = %s", (SOURCE_ITEM_ID,)))
    display(
        read_sql(
            """
            select source_item_id, ticker, source, published_at, summary, model,
                   embedded_at, vector_dims(embedding) as embedding_dimensions
            from item_embeddings
            where source_item_id = %s
            """,
            (SOURCE_ITEM_ID,),
        )
    )
    display(
        read_sql(
            """
            select *
            from item_connections
            where item_a_id = %s or item_b_id = %s
            order by confidence desc, verified_at desc
            """,
            (SOURCE_ITEM_ID, SOURCE_ITEM_ID),
        )
    )
else:
    print("Set SOURCE_ITEM_ID to trace a specific record.")

## Read-Only SQL Scratchpad

In [ ]:
CUSTOM_SQL = """
select ticker, source, count(*)::int as rows
from source_items
group by ticker, source
order by ticker, source
"""

read_sql(CUSTOM_SQL)